In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. knowledge base 구성을 위한 데이터 생성

In [6]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법_with_table.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 3.94 s
Wall time: 3.91 s


180

In [7]:
documents[43]

Document(metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천만원 초과 3억원 이하\n\n3,706만원 + (1억5천만원을 초과하는 금액의

In [1]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True) # 커널을 시작하지 않고 .env덮어쓰기
embedding = UpstageEmbeddings(
                model = 'embedding-query'
)

In [10]:
len(embedding.embed_query("소득세법"))

4096

In [24]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec
# 0. pinecone 객체 생성
pc = Pinecone() # api_key=os.getenv('PINECONE_API_KEY')

# 1. 인덱스 설정
index_name = 'tax-index-table'
dimension = len(embedding.embed_query("차원 확인용"))

# 2. 인덱스 생성(해당 인덱스가 없을 경우)
existing_indexes = [index.name for index in pc.list_indexes()]
# print(index_name in existing_indexes)
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name, 
        dimension=dimension,
        metric='cosine',  # 유사도 측정 방식
        spec=ServerlessSpec(
            cloud='aws',    # 무료 플랜은 aws
            region='us-east-1' # 무료플랜은 us-east-1만 가능
        )        
    )
    # 인덱스가 준비될 때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) # 1초 대기
    print(f'인덱스 {index_name}를 생성하였습니다')
else:
    print(f'인덱스 {index_name}이 이미 존재합니다')

인덱스 tax-index-table이 이미 존재합니다


In [28]:
# 3. 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['tax-index-openai', 'tax-index-table', 'tax-index']
True
{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [5]:
%%time
# 4. 문서를 임베딩해서 클라우드 vector DB(index)에 저장
# 처음 한 번
from pinecone import Pinecone
pc = Pinecone()
# from langchain_pinecone import PineconeVectorStore
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터DB를 load할 때
index_name = 'tax-index-table'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 0 ns
Wall time: 3.11 ms


# 2. 답변 생성 전 retrieval 확인

In [9]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, k=2) # 기본k값은 4

In [10]:
retrieved_docs[0]

Document(id='3a18b991-b767-4330-a4e1-8087fa3f0eb2', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천

In [28]:
retriever = database.as_retriever( 
    search_type='similarity',
    search_kwargs={'k':2}
)
retrieved_docs = retriever.invoke("연봉이 5천만원인 직장인의 소득세는 얼마인가요?")

In [15]:
retrieved_docs[0]

Document(id='3a18b991-b767-4330-a4e1-8087fa3f0eb2', metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천

In [29]:
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])
# print(retrieved_doc)

In [21]:
'55조' in retrieved_doc

True

# 3. 답변 생성

In [30]:
from openai import OpenAI
client = OpenAI()
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {'role':'system', 'content':'당신은 최고의 한국 소득세법 전문가입니다'},
        {
            'role':'user',
            'content':f"""- [context]를 참고해서 사용자의 질문에 답변해 주세요.
            - [context] : {retrieved_doc}
            - 질문 : {query}"""
        }
    ],
    temperature=0.2
)

In [31]:
result = response.choices[0].message.content
print(result)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 먼저 종합소득과세표준을 확인하고, 해당 세율을 적용해야 합니다.

1. **종합소득과세표준**: 연봉 5천만원은 과세표준에 해당합니다.

2. **세율 적용**:
   - 1,400만원 이하: 6% 
   - 1,400만원 초과 5,000만원 이하: 84만원 + (1,400만원을 초과하는 금액의 15%)

   연봉 5천만원에 대한 세액을 계산해보면:

   - 1,400만원에 대한 세액: 1,400만원 × 6% = 84만원
   - 5,000만원에서 1,400만원을 뺀 금액: 5,000만원 - 1,400만원 = 3,600만원
   - 3,600만원에 대한 세액: 3,600만원 × 15% = 540만원

3. **총 세액**:
   - 총 세액 = 84만원 + 540만원 = 624만원

따라서, 연봉 5천만원인 직장인의 소득세는 **624만원**입니다.


In [32]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-mini')

In [36]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model='solar-pro4',
    reasoning_effort='low' # 느리지만 더 깊게 추론하여 답변 (low, medium, high)
)

In [37]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다.
- [context]를 참고하여 사용자의 질문에 답변해 주세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [38]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

안녕하세요. 한국 소득세법 전문가로서 질문하신 내용에 대해 답변드리겠습니다.

질문하신 "연봉 5천만원인 직장인의 소득세"는 단순히 연봉만으로 확정되지 않으며, **소득공제**를 거쳐 결정된 **과세표준**에 세율을 적용하고, 다시 각종 **세액공제**를 차감하여 최종 세액이 결정됩니다.

제공해주신 법령 컨텍스트를 바탕으로 계산 구조와 추정치를 설명해 드리겠습니다.

### 1. 과세표준 계산 (소득에서 공제액을 뺌)
연봉 5천만원은 일반적으로 **총급여액**을 의미합니다. 여기서 소득세법 제47조 (근로소득공제) 등 각종 공제액을 빼야 비로소 세금 부과의 기준이 되는 **종합소득과세표준**이 나옵니다.
*   제공된 컨텍스트에는 구체적인 근로소득공제율이 명시되어 있지 않으나, 일반적으로 5천만원 수준의 총급여액이라면 약 13%~14% 수준의 근로소득공제와 인적공제 (본인, 부양가족 등) 가 적용되어 과세표준은 총급여에 비해 낮아집니다.
*   **가정:** 만약 다른 공제가 없다고 가정하고 순수하게 **과세표준이 5,000만원**이라고 가정하여 세율을 적용해 보겠습니다.

### 2. 산출세액 계산 (제55조 세율 적용)
제공된 **제55조(세율)** 표에 따르면, 과세표준 5,000만원은 두 번째 구간에 해당합니다.
*   **구간:** 1,400만원 초과 5,000만원 이하
*   **세액 계산식:** 84만원 + (1,400만원을 초과하는 금액의 15%)
*   **계산:** 840,000원 + (50,000,000원 - 14,000,000원) × 15%
    *   = 840,000원 + (36,000,000원 × 0.15)
    *   = 840,000원 + 5,400,000원
    *   = **6,240,000원** (산출세액)

### 3. 세액공제 적용 (제59조 등)
산출세액이 나왔다고 해서 당장 내야 하는 세금이 그 금액은 아닙니다. 각종 세액공제를 차감해야 합니다.

**가. 근로소득세액공제 (제59조)**
제공된 **제59조(근

## Langchain

In [1]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')
embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')

# 2. vector DB load
# pc = Pinecone()
index_name = 'tax-index-table'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 3. Retriever 생성 - search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':2}) # 유사도가 높은 top-2추출

# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [2]:
# 6. RAG 렝체인 구성 (LCEL방식)
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

종합소득세는 총급여가 아닌 근로소득공제 적용 후 계산된 과세표준에 세율을 부과하므로, 제공된 세율표만으로는 연봉을 과세표준으로 전환하지 못해 정확한 세액이 산출되지 않습니다. 또한 제시된 문맥에는 근로소득공제나 기본공제 등 해당 근로자에게 적용될 구체적 공제액과 세액공제 규모가 명시되지 않아 단정할 수 없습니다. 따라서 질문에 대한 정확한 답변은 문맥상 알 수 없습니다.
